# Find Homography So Dope — Complete Submission Notebook

**Objective**: Predict the $3 \times 3$ homography matrix $H$ (with $h_{33} = 1$) mapping pixel coordinates from `image_1` → `image_2` for each test pair.

$$
\begin{bmatrix} x_2' \\ y_2' \\ w_2' \end{bmatrix} = 
\begin{bmatrix} h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & 1 \end{bmatrix}
\begin{bmatrix} x_1 \\ y_1 \\ 1 \end{bmatrix}, \quad x_2 = \frac{x_2'}{w_2'}, \quad y_2 = \frac{y_2'}{w_2'}
$$

### Solution Pipeline
1. **Evaluation metric**: Geometric reprojection error over 5 canonical normalized coordinates
2. **EDA**: Sequence structure, identity homography discovery (illumination-only scenes)
3. **Multi-stage feature matching**: SIFT → RootSIFT → CLAHE contrast boost → USAC_MAGSAC robust estimation
4. **Geometric guardrails**: Determinant check, positive projective depth, convex quad, identity fallback
5. **Local validation**: Benchmarking on `train.csv`
6. **Visualization**: Keypoint matches & warped overlay alignment
7. **Test inference & submission**: Generating verified `submission.csv`

---
## 0. Setup & Imports

In [ ]:
import os
import sys
import time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Plotting defaults
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['image.interpolation'] = 'nearest'

print(f"Python  : {sys.version}")
print(f"OpenCV  : {cv2.__version__}")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")

In [ ]:
# ------------------------------------------------------------------
# PATH CONFIGURATION
# Auto-detects Kaggle vs local environment.
# On Kaggle: competition data is at /kaggle/input/find-homography-so-dope/
#            output goes to /kaggle/working/
# ------------------------------------------------------------------
KAGGLE_INPUT = '/kaggle/input/find-homography-so-dope'

if os.path.exists(KAGGLE_INPUT):
    # Running on Kaggle
    BASE_DIR   = KAGGLE_INPUT
    OUTPUT_CSV = '/kaggle/working/submission.csv'
    print('Environment: KAGGLE')
else:
    # Running locally
    BASE_DIR   = '.'
    OUTPUT_CSV = 'submission.csv'
    print('Environment: LOCAL')

TRAIN_CSV     = os.path.join(BASE_DIR, 'train.csv')
TEST_CSV      = os.path.join(BASE_DIR, 'test.csv')
TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'data', 'train')
TEST_IMG_DIR  = os.path.join(BASE_DIR, 'data', 'test')

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f"Train pairs: {len(train_df)}  (from {len(train_df) // 5} scenes)")
print(f"Test  pairs: {len(test_df)}  (from {len(test_df) // 5} scenes)")
print(f"\nTrain columns: {list(train_df.columns)}")
print(f"Test  columns: {list(test_df.columns)}")
print(f"\nTRAIN_IMG_DIR: {TRAIN_IMG_DIR}")
print(f"TEST_IMG_DIR:  {TEST_IMG_DIR}")
print(f"OUTPUT_CSV:    {OUTPUT_CSV}")

---
## 1. Evaluation Metric

Homographies are defined up to scale, so element-wise comparison is meaningless.  
We measure **geometric reprojection error**:
1. Take 5 normalized points in Image 1: $(0,0), (1,0), (1,1), (0,1), (0.5, 0.5)$
2. Convert to pixel coords using Image 1's $(W_1, H_1)$
3. Warp with predicted $H$ and ground-truth $H$
4. Convert back to normalized coords in Image 2 by dividing by $(W_2, H_2)$
5. Mean Euclidean distance across the 5 points

Kaggle Leaderboard Score: $\text{Score} = 100 \times \max\left(0, 1 - \dfrac{\text{Error}}{0.2}\right)$

In [ ]:
# 5 canonical normalized test points
POINTS_NORM = np.array([
    [0.0, 0.0],
    [1.0, 0.0],
    [1.0, 1.0],
    [0.0, 1.0],
    [0.5, 0.5]
], dtype=np.float64)


def warp_points(H: np.ndarray, pts: np.ndarray) -> np.ndarray:
    """Warp 2D points using a 3x3 homography matrix H."""
    pts_homo = np.hstack([pts, np.ones((len(pts), 1), dtype=np.float64)])
    warped = (H @ pts_homo.T).T
    w = warped[:, 2:3]
    w = np.where(np.abs(w) < 1e-8, 1e-8, w)
    return warped[:, :2] / w


def compute_reprojection_error(pred_h: np.ndarray, gt_h: np.ndarray,
                               w1: int, h1: int, w2: int, h2: int) -> float:
    """Mean reprojection error on 5 normalized points."""
    pts_px_1 = POINTS_NORM * np.array([w1, h1], dtype=np.float64)
    pred_px_2 = warp_points(pred_h, pts_px_1)
    gt_px_2   = warp_points(gt_h, pts_px_1)
    pred_norm_2 = pred_px_2 / np.array([w2, h2], dtype=np.float64)
    gt_norm_2   = gt_px_2   / np.array([w2, h2], dtype=np.float64)
    return float(np.linalg.norm(pred_norm_2 - gt_norm_2, axis=1).mean())


def lb_score_from_error(mean_err: float) -> float:
    """Convert reprojection error to Kaggle 0-100 Leaderboard Score."""
    return float(100.0 * max(0.0, 1.0 - mean_err / 0.2))


def parse_h(row) -> np.ndarray:
    """Parse homography matrix from a CSV row."""
    return np.array([
        [float(row['h11']), float(row['h12']), float(row['h13'])],
        [float(row['h21']), float(row['h22']), float(row['h23'])],
        [float(row['h31']), float(row['h32']), 1.0]
    ], dtype=np.float64)

---
## 2. Exploratory Data Analysis

Key discovery: ~50% of training scenes are **pure illumination changes** (camera did not move), so $H = I_{3\times 3}$.  
This means the identity matrix is already a strong prior baseline.

In [ ]:
# Detect pure identity homographies in training set
is_identity = (
    (train_df['h11'] == 1.0) & (train_df['h12'] == 0.0) & (train_df['h13'] == 0.0) &
    (train_df['h21'] == 0.0) & (train_df['h22'] == 1.0) & (train_df['h23'] == 0.0) &
    (train_df['h31'] == 0.0) & (train_df['h32'] == 0.0)
)
n_id = is_identity.sum()
print(f"Pure Identity pairs in Train: {n_id} / {len(train_df)} ({n_id / len(train_df) * 100:.1f}%)")

# Count unique scenes and their type
train_df['scene'] = train_df['pair_id'].str.rsplit('_', n=2).str[0] + '_' + train_df['pair_id'].str.rsplit('_', n=2).str[1]
scene_types = train_df.groupby('scene').apply(lambda g: 'illumination' if (g[['h11','h12','h13','h21','h22','h23','h31','h32']].values == np.array([1,0,0,0,1,0,0,0])).all() else 'viewpoint')
print(f"\nScene breakdown:")
print(scene_types.value_counts())

In [ ]:
# Compute the Identity Baseline score on the training set
errors_identity = []
for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Identity Baseline"):
    p1 = os.path.join(TRAIN_IMG_DIR, row['image_1'])
    p2 = os.path.join(TRAIN_IMG_DIR, row['image_2'])
    im1 = cv2.imread(p1)
    im2 = cv2.imread(p2)
    h1, w1 = im1.shape[:2]
    h2, w2 = im2.shape[:2]
    H_gt = parse_h(row)
    errors_identity.append(compute_reprojection_error(np.eye(3), H_gt, w1, h1, w2, h2))

mean_err_id = np.mean(errors_identity)
print(f"\nIdentity Matrix Baseline on Train:")
print(f"  Mean Reprojection Error: {mean_err_id:.5f}")
print(f"  Leaderboard Match Score: {lb_score_from_error(mean_err_id):.2f} / 100")

In [ ]:
# Visualize per-pair error distribution of the identity baseline
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(errors_identity, bins=50, color='#4c72b0', edgecolor='white', alpha=0.85)
axes[0].axvline(mean_err_id, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_err_id:.4f}')
axes[0].set_xlabel('Reprojection Error', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Identity Baseline — Error Distribution', fontsize=13)
axes[0].legend(fontsize=11)

# Show image dimensions across training scenes
dims = []
for _, row in train_df.drop_duplicates('image_1').iterrows():
    im = cv2.imread(os.path.join(TRAIN_IMG_DIR, row['image_1']))
    if im is not None:
        dims.append((im.shape[1], im.shape[0]))
dims = np.array(dims)
axes[1].scatter(dims[:, 0], dims[:, 1], c='#55a868', s=60, alpha=0.7, edgecolors='white')
axes[1].set_xlabel('Width (px)', fontsize=12)
axes[1].set_ylabel('Height (px)', fontsize=12)
axes[1].set_title('Image Dimensions (unique Image 1s)', fontsize=13)

plt.tight_layout()
plt.show()

---
## 3. Multi-Stage Homography Estimation Pipeline

| Stage | Detector | Pre-processing | Descriptor | Lowe Ratio | RANSAC Thresh | Trigger |
|-------|----------|---------------|------------|------------|---------------|--------|
| 1 | SIFT (5K) | Raw grayscale | Standard SIFT | 0.75 | 3.0 | Always |
| 2 | SIFT (8K) | CLAHE (clip 2.5) | **RootSIFT** | 0.75 | 3.0 | inliers < 15 |
| 3 | SIFT (8K) | CLAHE (clip 2.5) | Standard SIFT | 0.80 | 5.0 | inliers < 10 |
| 4 | SIFT (10K) | CLAHE (clip 3.5) | Standard SIFT | 0.85 | 5.0 | inliers < 10 |
| Fallback | — | — | — | — | — | inliers < 10 → $I_{3\times 3}$ |

**Geometric guardrails** at every stage:
- Determinant: $0.005 < |\det(H)| < 50$
- Positive projective depth: $w > 0$ for all 4 warped corners
- Convex quadrilateral check (no self-intersection)
- Bounding box sanity (not degenerate, not exploded)
- Robust estimator: **USAC_MAGSAC** (superior to classic RANSAC)

In [ ]:
# ======================================================================
# GEOMETRIC VALIDATION UTILITIES
# ======================================================================

def is_convex_quad(pts: np.ndarray) -> bool:
    """Check if 4 points form a strictly convex polygon."""
    cross_products = []
    for i in range(4):
        p1 = pts[i]
        p2 = pts[(i + 1) % 4]
        p3 = pts[(i + 2) % 4]
        v1 = p2 - p1
        v2 = p3 - p2
        cp = v1[0] * v2[1] - v1[1] * v2[0]
        cross_products.append(cp)
    return all(cp > 0 for cp in cross_products) or all(cp < 0 for cp in cross_products)


def is_valid_homography(H: np.ndarray, w1: int, h1: int, w2: int, h2: int) -> bool:
    """
    Validate physical feasibility of the homography:
    - Finite values
    - Determinant within realistic bounds
    - Positive projective depth (w > 0) for all warped corners
    - Warped boundary forms a convex quadrilateral
    - Warped bounding box within reasonable range
    """
    if H is None or not np.isfinite(H).all():
        return False

    det = np.linalg.det(H)
    if not (0.005 < abs(det) < 50.0):
        return False

    corners_1 = np.array([
        [0.0, 0.0, 1.0],
        [float(w1), 0.0, 1.0],
        [float(w1), float(h1), 1.0],
        [0.0, float(h1), 1.0]
    ], dtype=np.float64)

    warped_homo = (H @ corners_1.T).T
    if (warped_homo[:, 2] <= 1e-4).any():
        return False

    warped_pts = warped_homo[:, :2] / warped_homo[:, 2:3]
    if not is_convex_quad(warped_pts):
        return False

    x_min, y_min = warped_pts.min(axis=0)
    x_max, y_max = warped_pts.max(axis=0)
    box_w = x_max - x_min
    box_h = y_max - y_min
    if box_w < 10 or box_h < 10:
        return False
    if box_w > 20 * w2 or box_h > 20 * h2:
        return False

    return True

In [ ]:
# ======================================================================
# FEATURE EXTRACTION & DESCRIPTOR TRANSFORMS
# ======================================================================

def rootsift(des: np.ndarray) -> np.ndarray:
    """
    RootSIFT: Hellinger kernel normalization.
    L1-normalize -> sqrt -> L2-normalize.
    Dramatically reduces false matches vs. raw SIFT descriptors.
    """
    if des is None or len(des) == 0:
        return des
    des_norm = des / (np.linalg.norm(des, axis=1, ord=1, keepdims=True) + 1e-7)
    des_sqrt = np.sqrt(des_norm)
    des_l2   = des_sqrt / (np.linalg.norm(des_sqrt, axis=1, ord=2, keepdims=True) + 1e-7)
    return des_l2.astype(np.float32)


# Pre-initialize detectors and matchers (created once, reused for every pair)
sift_standard  = cv2.SIFT_create(nfeatures=5000,  contrastThreshold=0.015, edgeThreshold=10)
sift_sensitive  = cv2.SIFT_create(nfeatures=8000,  contrastThreshold=0.008, edgeThreshold=10)
sift_aggressive = cv2.SIFT_create(nfeatures=10000, contrastThreshold=0.005, edgeThreshold=10)
bf_matcher      = cv2.BFMatcher(cv2.NORM_L2)
clahe_filter    = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
clahe_strong    = cv2.createCLAHE(clipLimit=3.5, tileGridSize=(8, 8))

print("Detectors and matchers initialized.")

In [ ]:
# ======================================================================
# MATCHING + ROBUST ESTIMATION
# ======================================================================

def match_and_estimate(kp1, des1, kp2, des2,
                       w1, h1, w2, h2,
                       ratio=0.75, ransac_thresh=3.0):
    """
    Lowe ratio test -> USAC_MAGSAC homography estimation -> validation.
    Returns (H, inlier_count) or (None, 0) on failure.
    """
    if des1 is None or des2 is None or len(des1) < 4 or len(des2) < 4:
        return None, 0

    matches = bf_matcher.knnMatch(des1, des2, k=2)
    good = [m for m, n in matches if len((m, n)) == 2 and m.distance < ratio * n.distance]
    if len(good) < 4:
        return None, 0

    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.USAC_MAGSAC, ransac_thresh)
    inliers = int(mask.sum()) if mask is not None else 0

    if H is not None and np.isfinite(H).all() and abs(H[2, 2]) > 1e-8:
        H_norm = H / H[2, 2]
        if is_valid_homography(H_norm, w1, h1, w2, h2):
            return H_norm, inliers

    return None, 0

In [ ]:
# ======================================================================
# MAIN PREDICT FUNCTION  (This is also the function used for benchmarking)
# ======================================================================

def predict(image_1_path: str, image_2_path: str) -> np.ndarray:
    """
    Predict the 3x3 homography matrix mapping Image 1 -> Image 2.
    Returns np.ndarray of shape (3, 3) with H[2, 2] == 1.0.
    """
    img1 = cv2.imread(image_1_path, cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(image_2_path, cv2.IMREAD_GRAYSCALE)
    if img1 is None or img2 is None:
        return np.eye(3, dtype=np.float64)

    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    # ---- Stage 1: Standard SIFT, strict ratio 0.75 ----
    kp1, des1 = sift_standard.detectAndCompute(img1, None)
    kp2, des2 = sift_standard.detectAndCompute(img2, None)
    H_pred, inliers = match_and_estimate(
        kp1, des1, kp2, des2, w1, h1, w2, h2, ratio=0.75, ransac_thresh=3.0
    )

    # ---- Stage 2: CLAHE + RootSIFT (tough lighting) ----
    if H_pred is None or inliers < 15:
        img1_c = clahe_filter.apply(img1)
        img2_c = clahe_filter.apply(img2)
        kp1_c, des1_c = sift_sensitive.detectAndCompute(img1_c, None)
        kp2_c, des2_c = sift_sensitive.detectAndCompute(img2_c, None)
        H_c, inliers_c = match_and_estimate(
            kp1_c, rootsift(des1_c), kp2_c, rootsift(des2_c),
            w1, h1, w2, h2, ratio=0.75, ransac_thresh=3.0
        )
        if H_c is not None and inliers_c > max(inliers, 0):
            H_pred, inliers = H_c, inliers_c

    # ---- Stage 3: CLAHE + standard SIFT, relaxed ratio 0.80 ----
    if H_pred is None or inliers < 10:
        img1_c = clahe_filter.apply(img1)
        img2_c = clahe_filter.apply(img2)
        kp1_c, des1_c = sift_sensitive.detectAndCompute(img1_c, None)
        kp2_c, des2_c = sift_sensitive.detectAndCompute(img2_c, None)
        H_c, inliers_c = match_and_estimate(
            kp1_c, des1_c, kp2_c, des2_c,
            w1, h1, w2, h2, ratio=0.80, ransac_thresh=5.0
        )
        if H_c is not None and inliers_c > max(inliers, 0):
            H_pred, inliers = H_c, inliers_c

    # ---- Stage 4: Aggressive CLAHE + ultra-sensitive SIFT ----
    if H_pred is None or inliers < 10:
        img1_c = clahe_strong.apply(img1)
        img2_c = clahe_strong.apply(img2)
        kp1_c, des1_c = sift_aggressive.detectAndCompute(img1_c, None)
        kp2_c, des2_c = sift_aggressive.detectAndCompute(img2_c, None)
        H_c, inliers_c = match_and_estimate(
            kp1_c, des1_c, kp2_c, des2_c,
            w1, h1, w2, h2, ratio=0.85, ransac_thresh=5.0
        )
        if H_c is not None and inliers_c > max(inliers, 0):
            H_pred, inliers = H_c, inliers_c

    # ---- Fallback: Identity prior ----
    if H_pred is None or inliers < 10:
        H_pred = np.eye(3, dtype=np.float64)

    # Ensure exact H[2,2] == 1.0
    H_pred = H_pred / H_pred[2, 2]
    return H_pred.astype(np.float64)

---
## 4. Local Validation on Training Set

Run the full pipeline on all 140 training pairs to measure performance.

In [ ]:
train_errors = []
train_predictions = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Evaluating Train Pairs"):
    p1 = os.path.join(TRAIN_IMG_DIR, row['image_1'])
    p2 = os.path.join(TRAIN_IMG_DIR, row['image_2'])
    im1 = cv2.imread(p1)
    im2 = cv2.imread(p2)
    h1, w1 = im1.shape[:2]
    h2, w2 = im2.shape[:2]

    H_gt   = parse_h(row)
    H_pred = predict(p1, p2)
    err    = compute_reprojection_error(H_pred, H_gt, w1, h1, w2, h2)

    train_errors.append(err)
    train_predictions.append({
        'pair_id': row['pair_id'],
        'h11': H_pred[0,0], 'h12': H_pred[0,1], 'h13': H_pred[0,2],
        'h21': H_pred[1,0], 'h22': H_pred[1,1], 'h23': H_pred[1,2],
        'h31': H_pred[2,0], 'h32': H_pred[2,1],
        'error': err
    })

mean_train_err = np.mean(train_errors)
train_score = lb_score_from_error(mean_train_err)

print(f"\n{'='*50}")
print(f"  VALIDATION BENCHMARK RESULTS")
print(f"{'='*50}")
print(f"  Mean Reprojection Error : {mean_train_err:.5f}  (lower is better)")
print(f"  Kaggle Leaderboard Score: {train_score:.2f} / 100  (higher is better)")
print(f"  Improvement over Identity baseline: +{train_score - lb_score_from_error(mean_err_id):.2f} pts")
print(f"{'='*50}")

In [ ]:
# Per-pair error analysis
pred_analysis = pd.DataFrame(train_predictions)
pred_analysis['scene'] = pred_analysis['pair_id'].str.rsplit('_', n=2).str[0] + '_' + pred_analysis['pair_id'].str.rsplit('_', n=2).str[1]

# Show worst pairs (highest error)
print("\n--- Top 10 Worst Pairs (highest reprojection error) ---")
print(pred_analysis.nlargest(10, 'error')[['pair_id', 'error']].to_string(index=False))

# Error distribution: our pipeline vs identity
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(errors_identity, bins=50, alpha=0.5, color='#dd8452', label=f'Identity (score={lb_score_from_error(mean_err_id):.1f})', edgecolor='white')
ax.hist(train_errors, bins=50, alpha=0.6, color='#4c72b0', label=f'Our Pipeline (score={train_score:.1f})', edgecolor='white')
ax.axvline(mean_train_err, color='blue', linestyle='--', linewidth=2)
ax.axvline(mean_err_id, color='orange', linestyle='--', linewidth=2)
ax.set_xlabel('Reprojection Error', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Error Distribution: Identity Baseline vs. Multi-Stage Pipeline', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 5. Qualitative Visualization

Visual inspection on a viewpoint-shifted pair: detected inlier matches and warped overlay.

In [ ]:
# Pick a viewpoint pair for visualization
viewpoint_pairs = train_df[~is_identity]
sample_row = viewpoint_pairs.iloc[0]  # first viewpoint pair
p1 = os.path.join(TRAIN_IMG_DIR, sample_row['image_1'])
p2 = os.path.join(TRAIN_IMG_DIR, sample_row['image_2'])

im1_bgr = cv2.imread(p1)
im2_bgr = cv2.imread(p2)
h1_vis, w1_vis = im1_bgr.shape[:2]
h2_vis, w2_vis = im2_bgr.shape[:2]

# Predict homography
H_vis = predict(p1, p2)

# Ground truth
H_gt_vis = parse_h(sample_row)
err_vis = compute_reprojection_error(H_vis, H_gt_vis, w1_vis, h1_vis, w2_vis, h2_vis)

print(f"Pair: {sample_row['pair_id']}")
print(f"Reprojection Error: {err_vis:.6f}")
print(f"\nPredicted H:\n{H_vis}")
print(f"\nGround Truth H:\n{H_gt_vis}")

In [ ]:
# Warp Image 1 onto Image 2's canvas and create alpha-blend overlay
warped_im1 = cv2.warpPerspective(im1_bgr, H_vis, (w2_vis, h2_vis))
overlay = cv2.addWeighted(im2_bgr, 0.5, warped_im1, 0.5, 0)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

axes[0].imshow(cv2.cvtColor(im1_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title('Image 1 (Reference)', fontsize=13)
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(im2_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title('Image 2 (Target)', fontsize=13)
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[2].set_title(f'Aligned Overlay (err={err_vis:.5f})', fontsize=13)
axes[2].axis('off')

plt.suptitle(f'Pair: {sample_row["pair_id"]}', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Show SIFT keypoint matches for the same pair
img1_gray = cv2.imread(p1, cv2.IMREAD_GRAYSCALE)
img2_gray = cv2.imread(p2, cv2.IMREAD_GRAYSCALE)

kp1_v, des1_v = sift_standard.detectAndCompute(img1_gray, None)
kp2_v, des2_v = sift_standard.detectAndCompute(img2_gray, None)

matches_v = bf_matcher.knnMatch(des1_v, des2_v, k=2)
good_v = [m for m, n in matches_v if m.distance < 0.75 * n.distance]

# Draw top 50 matches
match_img = cv2.drawMatches(
    img1_gray, kp1_v, img2_gray, kp2_v,
    good_v[:50], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    matchColor=(0, 255, 0)
)

fig, ax = plt.subplots(figsize=(18, 7))
ax.imshow(match_img)
ax.set_title(f'SIFT Inlier Matches ({len(good_v)} total, showing top 50)', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 6. Test Set Inference & Submission Generation

Run inference on all test pairs and create `submission.csv` with format:  
`pair_id,h11,h12,h13,h21,h22,h23,h31,h32`

In [ ]:
submission_rows = []
start_time = time.time()

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test Inference"):
    pair_id = row['pair_id']
    p1 = os.path.join(TEST_IMG_DIR, row['image_1'])
    p2 = os.path.join(TEST_IMG_DIR, row['image_2'])

    H = predict(p1, p2)

    submission_rows.append({
        'pair_id': pair_id,
        'h11': float(H[0, 0]),
        'h12': float(H[0, 1]),
        'h13': float(H[0, 2]),
        'h21': float(H[1, 0]),
        'h22': float(H[1, 1]),
        'h23': float(H[1, 2]),
        'h31': float(H[2, 0]),
        'h32': float(H[2, 1])
    })

elapsed = time.time() - start_time

sub_df = pd.DataFrame(submission_rows)
cols = ['pair_id', 'h11', 'h12', 'h13', 'h21', 'h22', 'h23', 'h31', 'h32']
sub_df = sub_df[cols]

sub_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved submission to: {OUTPUT_CSV}")
print(f"Total inference time: {elapsed:.2f}s  ({elapsed / len(test_df):.3f}s / pair)")
print(f"Rows: {len(sub_df)}, Columns: {list(sub_df.columns)}")
sub_df.head(10)

---
## 7. Submission Validation

Final checks before Kaggle upload:
- Column names match expected format
- Row count matches `test.csv`
- `pair_id` values and order match exactly
- No NaN/Inf values
- All homography matrices are invertible (non-zero determinant)

In [ ]:
# ======================================================================
# FULL SUBMISSION VALIDATION
# ======================================================================

EXPECTED_COLS = ['pair_id', 'h11', 'h12', 'h13', 'h21', 'h22', 'h23', 'h31', 'h32']

sub_check = pd.read_csv(OUTPUT_CSV)
test_check = pd.read_csv(TEST_CSV)

all_ok = True

# 1. Column check
if list(sub_check.columns) == EXPECTED_COLS:
    print("  [OK] Column headers match expected format.")
else:
    print(f"  [ERROR] Columns mismatch! Expected {EXPECTED_COLS}, got {list(sub_check.columns)}")
    all_ok = False

# 2. Row count
if len(sub_check) == len(test_check):
    print(f"  [OK] Row count matches ({len(sub_check)} rows).")
else:
    print(f"  [ERROR] Row count mismatch! Expected {len(test_check)}, got {len(sub_check)}.")
    all_ok = False

# 3. pair_id match
if (sub_check['pair_id'].values == test_check['pair_id'].values).all():
    print("  [OK] pair_id values and order match test.csv perfectly.")
else:
    print("  [ERROR] pair_id values or order do not match test.csv!")
    all_ok = False

# 4. NaN / Inf check
has_null = False
has_inf = False
for col in EXPECTED_COLS[1:]:
    if sub_check[col].isnull().any():
        print(f"  [ERROR] Column {col} contains NaN values!")
        has_null = True
    if np.isinf(sub_check[col]).any():
        print(f"  [ERROR] Column {col} contains Inf values!")
        has_inf = True
if not has_null and not has_inf:
    print("  [OK] No nulls, NaNs, or Infs found.")
else:
    all_ok = False

# 5. Non-degeneracy (determinant check)
degenerate = 0
for _, r in sub_check.iterrows():
    H_check = np.array([
        [float(r['h11']), float(r['h12']), float(r['h13'])],
        [float(r['h21']), float(r['h22']), float(r['h23'])],
        [float(r['h31']), float(r['h32']), 1.0]
    ])
    det = np.linalg.det(H_check)
    if abs(det) < 1e-8:
        degenerate += 1
        print(f"  [WARNING] Pair {r['pair_id']} has near-zero det ({det:.2e})!")

if degenerate == 0:
    print("  [OK] All homography matrices are non-singular and invertible.")
else:
    print(f"  [WARNING] Found {degenerate} singular/degenerate matrices.")

if all_ok:
    print("\n" + "="*60)
    print("  SUCCESS: submission.csv is valid and ready for Kaggle upload!")
    print("="*60)
else:
    print("\n  FAILED: Fix the errors above before submitting.")